# Cold Snap Recovery Burden
### Metric: MAE

**Key findings:**
- Test cities are fully OOD (0 location overlap with train)
- Target spans 170–11866 with skew=1.56; raw MAE loss outperforms log target
- Highest individual correlations: relhum_min (r=0.25), dew_dry_spread (r=-0.21), ghi_deficit
- `prefix_hours` is a critical conditioning feature (median target: 12h=1442, 24h=1048, 36h=501)
- Top LGB features: dayofyear, pressure_slope, windspeed_slope, lat_x_month_sin

In [48]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('scikit-learn')

In [49]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from scipy.optimize import minimize

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

In [50]:
PLATFORM_DATA_DIR = Path('dataset/public')
if PLATFORM_DATA_DIR.exists():
    TRAIN_PATH  = PLATFORM_DATA_DIR / 'train.csv'
    TEST_PATH   = PLATFORM_DATA_DIR / 'test.csv'
    SUBMIT_PATH = Path('working/submission.csv')
    Path('working').mkdir(exist_ok=True)
else:
    from google.colab import files
    files.upload()
    TRAIN_PATH  = Path('train.csv')
    TEST_PATH   = Path('test.csv')
    SUBMIT_PATH = Path('submission.csv')

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')

Saving train.csv to train (6).csv
Saving test.csv to test (6).csv
Saving sample_submission.csv to sample_submission (6).csv
Train: (1193, 46) | Test: (538, 45)


In [72]:
TARGET = 'remaining_hdh'
SEED   = 42
N_FOLDS = 5
SEEDS  = [42, 7, 123]

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [116]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

In [73]:
# ── Feature engineering ────────────────────────────────────────────────────────
def build_features(df):
    X = df.drop(columns=['id', TARGET], errors='ignore').copy()

    # Heating rate during observed prefix
    X['hdh_rate']          = X['prefix_hdh'] / X['prefix_hours']

    # Current thermal deficit relative to 18C base
    X['temp_deficit']      = 18 - X['drybulb_last']

    # Trajectory signals
    X['warming_signal']    = (X['drybulb_slope'] > 0).astype(int)
    X['cold_deepening']    = (X['drybulb_slope'] < 0).astype(int) * X['temp_deficit']
    X['cold_front']        = ((X['pressure_slope'] > 0) & (X['drybulb_slope'] < 0)).astype(int)

    # Temperature extrapolations
    X['drybulb_extrap_24'] = X['drybulb_last'] + X['drybulb_slope'] * 24
    X['drybulb_extrap_48'] = X['drybulb_last'] + X['drybulb_slope'] * 48

    # Burden proxy: project current rate over remaining window
    X['prefix_frac']       = X['prefix_hours'] / 36
    X['extrap_remaining']  = X['hdh_rate'] * (48 - X['prefix_hours'])

    # Composite weather signals
    X['wind_chill']        = X['windspeed_mean'] * X['temp_deficit']
    X['temp_x_relhum']     = X['temp_deficit']   * X['relhum_mean']
    X['slope_x_deficit']   = X['drybulb_slope']  * X['temp_deficit']
    X['dew_dry_spread']    = X['drybulb_last']   - X['dewpoint_last']
    X['pressure_range']    = X['pressure_max']   - X['pressure_min']
    X['ghi_deficit']       = X['ghi_mean']       * (18 - X['drybulb_mean'])
    X['relhum_x_wind']     = X['relhum_mean']    * X['windspeed_mean']
    X['hdh_rate_x_lat']    = X['hdh_rate']       * X['latitude']

    # Geographic context
    X['lat_sq']            = X['latitude'] ** 2
    X['lat_x_elev']        = X['latitude']  * X['elevation_m']

    # Cyclic time encoding
    X['month_sin']         = np.sin(2 * np.pi * X['month']     / 12)
    X['month_cos']         = np.cos(2 * np.pi * X['month']     / 12)
    X['doy_sin']           = np.sin(2 * np.pi * X['dayofyear'] / 365)
    X['doy_cos']           = np.cos(2 * np.pi * X['dayofyear'] / 365)
    X['lat_x_month_sin']   = X['latitude'] * X['month_sin']

    return X


train_fe = build_features(train_raw)
test_fe  = build_features(test_raw)

FEATURE_COLS = train_fe.columns.tolist()
print(f'Features: {len(FEATURE_COLS)}')

X_all    = train_fe.values
y_all    = train_raw[TARGET].values
X_test   = test_fe.values

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

Features: 68


In [85]:
# ── Neural Network ─────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.act(x + self.net(x))


class BurdenNet(nn.Module):
    def __init__(self, n_features, hidden=512, n_blocks=4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU()
        )
        self.blocks = nn.Sequential(*[ResBlock(hidden) for _ in range(n_blocks)])
        self.output  = nn.Sequential(
            nn.Linear(hidden, 128), nn.SiLU(),
            nn.Linear(128, 1), nn.Softplus()
        )
    def forward(self, x):
        return self.output(self.blocks(self.input_proj(x))).squeeze(-1)

In [104]:
# ── BranchedNet — rama por grupo de features ──────────────────────────────────
FEATURE_GROUPS = {
    'temperature': [1,2,3,4,5,6,7,8,9,10,11,12,45,56,46,47,49,50,55],
    'atmospheric': [26,27,28,29,30,31,57,38,39,40,41,42,43,48,53],
    'solar_humid': [14,15,16,17,18,19,32,33,34,35,36,37,58,59,54],
    'geo_time':    [20,21,13,22,0,63,64,65,66,61,62,67,60],
    'prefix_dyn':  [23,24,25,51,44,52],
}

class BranchedNet(nn.Module):
    def __init__(self, branch_hidden=64, trunk_hidden=512, n_trunk_blocks=4, dropout=0):
        super().__init__()
        self.groups = FEATURE_GROUPS
        self.branches = nn.ModuleDict({
            name: nn.Sequential(
                nn.Linear(len(idxs), branch_hidden), nn.BatchNorm1d(branch_hidden), nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(branch_hidden, branch_hidden), nn.BatchNorm1d(branch_hidden), nn.SiLU(),
            )
            for name, idxs in self.groups.items()
        })
        trunk_in = branch_hidden * len(self.groups)
        self.trunk = nn.Sequential(
            nn.Linear(trunk_in, trunk_hidden), nn.BatchNorm1d(trunk_hidden), nn.SiLU(),
            *[ResBlock(trunk_hidden, dropout=dropout) for _ in range(n_trunk_blocks)],
        )
        self.output = nn.Sequential(
            nn.Linear(trunk_hidden, 512), nn.SiLU(),
            nn.Linear(512, 1), nn.Softplus()
        )

    def forward(self, x):
        branch_outs = [
            self.branches[name](x[:, idxs])
            for name, idxs in self.groups.items()
        ]
        return self.output(self.trunk(torch.cat(branch_outs, dim=1))).squeeze(-1)

In [87]:
def make_loader(X, y=None, batch_size=256, shuffle=False):
    Xt = torch.tensor(X, dtype=torch.float32)
    if y is not None:
        return DataLoader(TensorDataset(Xt, torch.tensor(y, dtype=torch.float32)),
                          batch_size=batch_size, shuffle=shuffle)
    return DataLoader(TensorDataset(Xt), batch_size=batch_size)


def train_nn(X_tr, y_tr, X_va, y_va, epochs=500, lr=3e-4,
             hidden=512, n_blocks=4, model_class=BurdenNet):
    fs    = StandardScaler().fit(X_tr)
    model = model_class(X_tr.shape[1]).to(DEVICE) if model_class == BranchedNet \
            else model_class(X_tr.shape[1], hidden=hidden, n_blocks=n_blocks).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.L1Loss()

    best_mae   = 1e9
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    no_improve = 0
    tr_loader  = make_loader(fs.transform(X_tr), y_tr, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(Xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            Xv     = torch.tensor(fs.transform(X_va), dtype=torch.float32).to(DEVICE)
            val_mae = mae(y_va, model(Xv).cpu().numpy())
        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= 40:
            break

    model.load_state_dict(best_state)
    return model, fs


def predict_nn(model, fs, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(fs.transform(X), dtype=torch.float32).to(DEVICE)
        return model(Xt).cpu().numpy()

In [88]:
oof_nn  = np.zeros(len(X_all))
test_nn = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    model, fs = train_nn(
        X_all[tr_idx], y_all[tr_idx],
        X_all[va_idx], y_all[va_idx],
    )
    oof_nn[va_idx] = predict_nn(model, fs, X_all[va_idx])
    test_nn       += predict_nn(model, fs, X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {mae(y_all[va_idx], oof_nn[va_idx]):.4f}')

nn_mae = mae(y_all, np.clip(oof_nn, 0, None))
print(f'\nNN OOF MAE: {nn_mae:.4f}')

  Fold 1: 680.0390
  Fold 2: 698.2306
  Fold 3: 782.6526
  Fold 4: 687.8470
  Fold 5: 845.9151

NN OOF MAE: 738.8900


In [105]:
# ── BranchedNet CV ─────────────────────────────────────────────────────────────
oof_bn  = np.zeros(len(X_all))
test_bn = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    model, fs = train_nn(
        X_all[tr_idx], y_all[tr_idx],
        X_all[va_idx], y_all[va_idx],
        model_class=BranchedNet,
    )
    oof_bn[va_idx] = predict_nn(model, fs, X_all[va_idx])
    test_bn       += predict_nn(model, fs, X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {mae(y_all[va_idx], oof_bn[va_idx]):.4f}')

bn_mae = mae(y_all, np.clip(oof_bn, 0, None))
print(f'\nBranchedNet OOF MAE: {bn_mae:.4f}')

  Fold 1: 719.2098
  Fold 2: 867.3097
  Fold 3: 697.3812
  Fold 4: 783.9425
  Fold 5: 817.8714

BranchedNet OOF MAE: 777.1031


In [114]:
print('=' * 42)
print(f'NeuralNet OOF MAE : {nn_mae:.4f}')
print(f'Branched NeuralNet OOF MAE : {bn_mae:.4f}')
print('=' * 42)

NeuralNet OOF MAE : 738.8900
Branched NeuralNet OOF MAE : 777.1031


In [118]:
# ── Ensemble: OOF-optimized weights ───────────────────────────────────────────
oof_stack = np.stack([oof_bn, oof_nn], axis=1)

def ensemble_mae(w):
    w = np.abs(w) / np.abs(w).sum()
    return mae(y_all, np.clip((oof_stack * w).sum(axis=1), 0, None))

res   = minimize(ensemble_mae, x0=[1/2]*2, method='Nelder-Mead', options={'maxiter': 2000})
opt_w = np.abs(res.x) / np.abs(res.x).sum()
print(f'Weights — BN: {opt_w[0]:.3f} | NN: {opt_w[1]:.3f}')
print(f'Ensemble OOF MAE: {ensemble_mae(opt_w):.4f}')

Weights — BN: 0.385 | NN: 0.615
Ensemble OOF MAE: 717.4865


In [120]:
test_stack  = np.stack([test_bn, test_nn], axis=1)
final_preds = np.clip((test_stack * opt_w).sum(axis=1), 0, None)
print(f'Predictions: min={final_preds.min():.2f}, mean={final_preds.mean():.2f}, max={final_preds.max():.2f}')

Predictions: min=202.80, mean=2175.88, max=9242.94


In [122]:
submission = pd.DataFrame({'id': test_raw['id'], 'remaining_hdh': final_preds})
submission.to_csv(SUBMIT_PATH, index=False)
print(f'Saved: {SUBMIT_PATH}')
print(submission.head())

Saved: submission.csv
             id  remaining_hdh
0  dc3243cb79e9    1883.197864
1  0656227254b4    1046.501307
2  8b0f9859f31a     961.082005
3  339d514d5e0f    2915.782750
4  0fa70af05413    1992.036047
